# Cornucopia: logical-memory simulation

Prepare logical memory in the selected basis, apply noisy syndrome-extraction
cycles, and decode the final logical observables. The circuit noise satisfies
$p_{\mathrm{CX}}=p_{\mathrm{measurement}}=p_{\mathrm{reset}}=p$, including
final data measurement. No idle-noise channel is included.

`generate` writes circuit artifacts and detector/observable samples; `decode`
uses saved samples; `both` performs both stages. The `xz` mode retains the
CSS detector type relevant to the memory basis, while `xyz` retains all types.

Check `SHOTS` before running: these settings use large sample counts.
Outputs go to `circuit_simulation/results/`.


## Parameters


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "code_construction" / "affine_codes.py").is_file()
)
sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PYTHON = Path(sys.executable)
SCRIPT = REPO_ROOT / "circuit_simulation/simulate_memory.py"
SAMPLE_DIR = Path("circuit_simulation/results/samples")
DECODE_DIR = Path("circuit_simulation/results/decode")

STAGE = "both"  # generate | decode | both
DECODER = "relaybp"
# Code and physical noise.
CODES = ["cornucopia_p147_d14"]  # Use ["all"] to run every code.
BASIS = "X"  # Z | X | both
DECODING_MODE = "xz"  # xyz | xz | both
P_LIST = [0.002]

CYCLES = 7
SHOTS = 200_000  # per adaptive round; total shots extend by this amount

MIN_FAILURES = (
    0  # set 0 to disable adaptive extension; counts total failures including MIP fallback
)
SHOT_CHUNK = 200
WORKERS = max(1, min(8, os.cpu_count() or 1))
SEED = 20260807
DECODE_BATCH_SIZE = 100_000

# Primary RelayBP decoder.
GAMMA0 = 0.1
PRE_ITER = 200
NUM_SETS = 20
SET_MAX_ITER = 100
GAMMA_DIST_MIN = -0.24
GAMMA_DIST_MAX = 0.6
STOP_NCONV = 1


# Second RelayBP pass on unresolved shots.
RELAYBP_FALLBACK = True
RELAYBP_FALLBACK_RETRY_UNCONVERGED = (
    False  # set True only after a source decode has saved primary-unconverged details
)
RELAYBP_FALLBACK_SOURCE_HASH = ""
RELAYBP_FALLBACK_GAMMA0 = 0.1
RELAYBP_FALLBACK_PRE_ITER = 500
RELAYBP_FALLBACK_NUM_SETS = 200
RELAYBP_FALLBACK_SET_MAX_ITER = 200
RELAYBP_FALLBACK_GAMMA_DIST_MIN = -0.24
RELAYBP_FALLBACK_GAMMA_DIST_MAX = 0.6
RELAYBP_FALLBACK_STOP_NCONV = 1


# Optional mixed-integer decoding.
MIP_FALLBACK = False
MIP_TIME_LIMIT = 3000
MIP_WALL_TIME_LIMIT = 0
MIP_REL_GAP = 0.0
MIP_FEASIBLE_ONLY = False
MIP_WORKERS = 16
MIP_THREADS = 32
MIP_UNSOLVED_NOT_FAILURE = False
MIP_RETRY_UNSOLVED = False
MIP_RETRY_SOURCE_HASH = ""  # retry source with saved MIP-unsolved details

FORCE_ARTIFACTS = False
FORCE_SHOTS = False
FORCE_DECODE = False

print(f"workers={WORKERS}")
print(f"sample_dir={SAMPLE_DIR}")
print(f"decode_dir={DECODE_DIR}")

## Command


In [ ]:
def _csv(values):
    return ",".join(str(value) for value in values)


cmd = [
    str(PYTHON),
    str(SCRIPT),
    "--stage",
    STAGE,
    "--decoder",
    DECODER,
    "--codes",
    _csv(CODES),
    "--basis",
    BASIS,
    "--decoding-mode",
    DECODING_MODE,
    "--p-list",
    _csv(P_LIST),
    "--cycles",
    str(CYCLES),
    "--shots",
    str(SHOTS),
    "--shot-chunk",
    str(SHOT_CHUNK),
    "--decode-batch-size",
    str(DECODE_BATCH_SIZE),
    "--workers",
    str(WORKERS),
    "--gamma0",
    str(GAMMA0),
    "--pre-iter",
    str(PRE_ITER),
    "--num-sets",
    str(NUM_SETS),
    "--set-max-iter",
    str(SET_MAX_ITER),
    "--gamma-dist-min",
    str(GAMMA_DIST_MIN),
    "--gamma-dist-max",
    str(GAMMA_DIST_MAX),
    "--stop-nconv",
    str(STOP_NCONV),
    "--seed",
    str(SEED),
    "--sample-dir",
    str(SAMPLE_DIR),
    "--decode-dir",
    str(DECODE_DIR),
]

if RELAYBP_FALLBACK:
    cmd.append("--relaybp-fallback")
    cmd.extend(["--relaybp-fallback-gamma0", str(RELAYBP_FALLBACK_GAMMA0)])
    cmd.extend(["--relaybp-fallback-pre-iter", str(RELAYBP_FALLBACK_PRE_ITER)])
    cmd.extend(["--relaybp-fallback-num-sets", str(RELAYBP_FALLBACK_NUM_SETS)])
    cmd.extend(["--relaybp-fallback-set-max-iter", str(RELAYBP_FALLBACK_SET_MAX_ITER)])
    cmd.extend(["--relaybp-fallback-gamma-dist-min", str(RELAYBP_FALLBACK_GAMMA_DIST_MIN)])
    cmd.extend(["--relaybp-fallback-gamma-dist-max", str(RELAYBP_FALLBACK_GAMMA_DIST_MAX)])
    cmd.extend(["--relaybp-fallback-stop-nconv", str(RELAYBP_FALLBACK_STOP_NCONV)])
    if RELAYBP_FALLBACK_RETRY_UNCONVERGED:
        cmd.append("--relaybp-fallback-retry-unconverged")
        if str(RELAYBP_FALLBACK_SOURCE_HASH).strip():
            cmd.extend(
                ["--relaybp-fallback-source-hash", str(RELAYBP_FALLBACK_SOURCE_HASH).strip()]
            )

if MIN_FAILURES > 0:
    cmd.extend(["--min-failures", str(MIN_FAILURES)])

if MIP_FALLBACK:
    cmd.append("--mip-fallback")
    cmd.extend(["--mip-time-limit", str(MIP_TIME_LIMIT)])
    cmd.extend(["--mip-wall-time-limit", str(MIP_WALL_TIME_LIMIT)])
    cmd.extend(["--mip-rel-gap", str(MIP_REL_GAP)])
    if MIP_FEASIBLE_ONLY:
        cmd.append("--mip-feasible-only")
    cmd.extend(["--mip-workers", str(MIP_WORKERS)])
    cmd.extend(["--mip-threads", str(MIP_THREADS)])
    if MIP_UNSOLVED_NOT_FAILURE:
        cmd.append("--mip-unsolved-not-failure")

if RELAYBP_FALLBACK_RETRY_UNCONVERGED and not RELAYBP_FALLBACK:
    raise ValueError("RELAYBP_FALLBACK_RETRY_UNCONVERGED requires RELAYBP_FALLBACK")

if MIP_RETRY_UNSOLVED and RELAYBP_FALLBACK:
    raise ValueError("MIP_RETRY_UNSOLVED cannot be combined with RELAYBP_FALLBACK")

if MIP_RETRY_UNSOLVED:
    cmd.append("--mip-retry-unsolved")
    if str(MIP_RETRY_SOURCE_HASH).strip():
        cmd.extend(["--mip-retry-source-hash", str(MIP_RETRY_SOURCE_HASH).strip()])

if FORCE_ARTIFACTS:
    cmd.append("--force-artifacts")
if FORCE_SHOTS:
    cmd.append("--force-shots")
if FORCE_DECODE:
    cmd.append("--force-decode")

print(" ".join(["python", *cmd[1:]]))

## Run


In [ ]:
completed = subprocess.run(cmd, cwd=str(REPO_ROOT), text=True)
if completed.returncode:
    raise SystemExit(completed.returncode)

## Results


In [ ]:
for relative_path in [SAMPLE_DIR / "summary.txt", DECODE_DIR / "summary.txt"]:
    path = REPO_ROOT / relative_path
    print("====", path, "====")
    if path.exists():
        print(path.read_text())
    else:
        print("missing")